# Track B: Manufacturing Analysis (PrecisionWorks Industries)

Analysis of the case study Dataset B1 to B4. All data is illustrative, taken from the case study tables. Every number below is calculated in code from the CSV files in `data/`.

**Sections:** 1. Setup · 2. Line KPIs · 3. Machine risk · 4. Batch material variance · 5. Data checks · 6. Charts · 7. Findings

## 1. Setup
The notebook uses the `analysis.py` module from the prototype folder. Run it from inside `track_b_prototype/`.

In [ ]:
import analysis as a
import pandas as pd

lines, machines, batches = a.load_inputs()
print(f"Loaded {len(lines)} lines, {len(machines)} machines, {len(batches)} batches")
pd.DataFrame(lines)

Loaded 3 lines, 6 machines, 4 batches


,production_line,daily_target,actual_output,defective_units,downtime_hours_per_day
0,Line 1,1000.0,850.0,34.0,1.5
1,Line 2,1200.0,900.0,81.0,2.5
2,Line 3,800.0,720.0,22.0,1.0


## 2. Line KPIs
- **Achievement** = actual ÷ target
- **Defect rate** = defective ÷ actual
- **Downtime %** = downtime hours ÷ 16 scheduled hours
- **OEE (estimate)** = achievement × quality. Assumes target is the ideal rate. This is an estimate, not measured OEE.

In [ ]:
line_df = pd.DataFrame(a.line_kpis(lines))
show = line_df[["line","achievement","defect_rate","downtime_pct","oee_estimate","shortfall_units","downtime_loss_units","unexplained_shortfall_units"]].copy()
for c in ["achievement","defect_rate","downtime_pct","oee_estimate"]:
    show[c] = (show[c]*100).round(1).astype(str) + "%"
show.round(1)

,line,achievement,defect_rate,downtime_pct,oee_estimate,shortfall_units,downtime_loss_units,unexplained_shortfall_units
0,Line 1,85.0%,4.0%,9.4%,81.6%,150.0,93.8,56.2
1,Line 2,75.0%,9.0%,15.6%,68.2%,300.0,187.5,112.5
2,Line 3,90.0%,3.1%,6.2%,87.2%,80.0,50.0,30.0


**Reading the table:** Line 2 has the lowest achievement (75%), the highest defect rate (9.0%) and the most downtime (15.6%). It accounts for 300 of the 530 missing units. About 188 of those units are explained by downtime; the other ~112 are not, which points to speed losses or other causes that this data does not record.

## 3. Machine risk
A transparent weighted rule, not a trained model. Each signal is scaled to the highest value in the sample, then weighted: vibration 35%, temperature 25%, downtime 20%, defects 20%. With six machines and no failure labels, a trained model would not be justified.

In [ ]:
risk_df = pd.DataFrame(a.machine_risk(machines))
risk_df[["machine","risk_score","tier","temperature_c","vibration_mm_s","downtime_hours","defects"]].round(3)

,machine,risk_score,tier,temperature_c,vibration_mm_s,downtime_hours,defects
0,M004,1.000,High,91.0,6.2,3.0,29.0
1,M006,0.813,High,85.0,5.1,2.2,21.0
2,M002,0.754,High,82.0,4.8,2.0,18.0
3,M003,0.435,Low,70.0,2.5,0.8,7.0
4,M005,0.371,Low,68.0,2.3,0.4,4.0
5,M001,0.365,Low,65.0,2.1,0.5,5.0


**Reading the table:** M004, M006 and M002 are High risk. Together they account for 68 of 84 machine defects and 7.2 of 8.9 downtime hours. The machine scores are relative to this sample. They should be checked against the manufacturer's alarm limits, which the case study does not provide.

## 4. Batch material variance
Variance = (used − expected) ÷ expected. Batches are matched to lines by actual output (inferred, not stated in the case study).

In [ ]:
batch_df = pd.DataFrame(a.batch_material(batches))
batch_df.round(3)

,batch,used_kg,expected_kg,variance_kg,variance_pct,output,rejected
0,B001,520.0,500.0,20.0,0.040,850.0,34.0
1,B002,640.0,600.0,40.0,0.067,900.0,81.0
2,B003,410.0,400.0,10.0,0.025,720.0,22.0
3,B004,530.0,500.0,30.0,0.060,820.0,45.0


**Reading the table:** Total material use is 100 kg (5.0%) over expected. B002 is the worst at +6.7%. B004 matches no production line, so its origin is unknown.

## 5. Data checks
The brief asks us to identify gaps, inconsistencies and limitations. These checks are generated by code so they stay visible.

In [ ]:
for w in a.data_checks(lines, machines, batches):
    print("-", w)

- Downtime mismatch: lines total 5.0 h, machines total 8.9 h. Machines have no line ID, so they cannot be reconciled.
- Defect mismatch: lines total 137, machines total 84.
- Batch B004 (output 820) matches no production line. Its line of origin is unknown.
- Line data covers one day, but the brief's evaluation period is 30 days. No monthly figure can be computed from this data.
- Machine readings have no timestamps and arrive at irregular intervals, so they cannot be linked to specific downtime events.


## 6. Charts

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display  # shows the saved figures in Jupyter

fig_dir = Path("../figures")
for name in ["line_kpis.png", "machine_risk.png", "batch_variance.png"]:
    p = fig_dir / name
    if p.exists():
        display(Image(filename=str(p), width=600))

## 7. Findings
1. **Line 2 is the first priority.** Lowest achievement, highest defects and downtime.
2. **Three machines (M004, M006, M002) drive most of the defects and downtime.** Vibration and temperature rise with downtime and defects in this sample. This is a correlation across six machines, not proof of cause.
3. **Material use is 5% over expected**, concentrated in B002 and B004.
4. **The data does not reconcile.** Machine totals do not match line totals, so no single number should be trusted without checking the source.

**Next data needed:** timestamped machine readings, line IDs on each machine, downtime reasons, 30 days of daily production, and recorded intervention outcomes.